# 06 · Shortcut check: how much of the best model is the ICU clock?

Notebook 05 showed that ICULOS (hours since ICU admission) carries a lot of the model's performance. A model that mostly learns "the longer the stay, the higher the risk" is not reading the patient's physiology. That pattern may also not hold in another hospital. This notebook measures how big that effect is.

Same 5-fold patient-level CV on 80% of Hospital A and same weighted logistic regression. Four feature sets ("missingness" = the flags and hours-since features from notebook 04):

| feature set | inputs | question it answers |
|---|---|---|
| all + missingness | all 40 features + missingness | the best model so far: the reference |
| ICULOS only | one input: the hours-in-ICU counter | how far does the clock alone get? |
| all except ICULOS + missingness | the best model without the clock | what is lost without it? |
| clinical + missingness | vitals and labs with their missingness, no demographics or time | what does the physiology (and how it is monitored) give alone? |

In [1]:
import sepsis as s

data_A = s.load_hospital("A")
dev_ids, test_ids = s.split_patients(data_A, test_size=0.20)      # same split and folds asnotebooks 03-05
fold_of = s.make_folds(data_A, dev_ids)
dev = s.prepare_dev(data_A, dev_ids, fold_of)                     # with the missingness columns

In [2]:
no_iculos = [c for c in s.ALL_FEATURES if c != "ICULOS"]
missingness = dict(flags=True, since=True)

oofs = {
    "all + missingness": s.run_cv(dev, s.ALL_FEATURES, **missingness),      # reference
    "ICULOS only": s.run_cv(dev, ["ICULOS"]),                              # a single column: no missingness to add
    "all except ICULOS + missingness": s.run_cv(dev, no_iculos, **missingness),
    "clinical + missingness": s.run_cv(dev, s.CLINICAL, **missingness),
}

trained on 96 input features
trained on 96 input features
trained on 96 input features
trained on 96 input features
trained on 96 input features
trained on 1 input features
trained on 1 input features
trained on 1 input features
trained on 1 input features
trained on 1 input features
trained on 95 input features
trained on 95 input features
trained on 95 input features
trained on 95 input features
trained on 95 input features
trained on 90 input features
trained on 90 input features
trained on 90 input features
trained on 90 input features
trained on 90 input features


## 1. AUROC and AUPRC

The second table is each set minus the reference (*all + missingness*), fold by fold.

In [3]:
summary, diffs = s.compare_auc(oofs)
display(summary.round(4))
display(diffs.round(4))

AUROC           AUPRC        
                                   mean     std    mean     std
Features                                                       
all + missingness                0.7847  0.0232  0.0930  0.0159
ICULOS only                      0.6355  0.0269  0.0800  0.0188
all except ICULOS + missingness  0.7459  0.0225  0.0627  0.0089
clinical + missingness           0.7393  0.0209  0.0622  0.0079

AUROC                                                         \
Features ICULOS only all except ICULOS + missingness clinical + missingness   
1            -0.1655                         -0.0421                -0.0504   
2            -0.1505                         -0.0403                -0.0442   
3            -0.1500                         -0.0384                -0.0458   
4            -0.1217                         -0.0484                -0.0512   
5            -0.1585                         -0.0247                -0.0356   
mean         -0.1492                         -0.0388                -0.0454   
std           0.0166                          0.0087                 0.0062   

               AUPRC                                                         
Features ICULOS only all except ICULOS + missingness clinical + missingness  
1            -0.0265                         -0.0382                -0.0409  
2            -0.0161                         -0.0312                -0.0314  
3            -0.0124                         -0.0191                -0.0200  
4             0.0055                         -0.0407                -0.0395  
5            -0.0157                         -0.0224                -0.0219  
mean         -0.0130                         -0.0303                -0.0308  
std           0.0117                          0.0094                 0.0097

## 2. Each feature set at its own best cutoff

In [4]:
s.best_cutoff_table(oofs).round(4)

,Best cutoff,F1,Precision,Recall,FPR,Clinical utility,"Utility, fold mean","Utility, fold std"
Features,,,,,,,,
all + missingness,0.55,0.1179,0.0652,0.6171,0.1966,0.3674,0.3673,0.0428
ICULOS only,0.55,0.1519,0.0938,0.4004,0.0859,0.2994,0.2993,0.0419
all except ICULOS + missingness,0.55,0.0952,0.0517,0.5991,0.2439,0.2863,0.2863,0.0407
clinical + missingness,0.55,0.0928,0.0504,0.5885,0.2464,0.2715,0.2715,0.0385


## Conclusion

The clock alone goes surprisingly far. With ICULOS as its only input, the model reaches an AUROC of about 0.64. Its AUPRC is 0.080, close to the best model's 0.093. Its best clinical utility is 0.30, higher than the 0.27 of all vitals and labs with their missingness features (clinical + missingness).

Removing the clock from the best model costs about 0.04 AUROC and 0.03 AUPRC, in every fold. The utility drops from 0.37 to 0.29. Without ICULOS, the model is almost the same as one with physiology and monitoring patterns only, so the demographic variables other than ICULOS add little.

The reason is visible in notebook 01, section 5: the share of positive labels rises with time in the ICU. That is partly real (a longer stay gives more time to develop sepsis), but it says when sepsis is likely, not whether this patient is deteriorating. It also depends on how each hospital admits and labels patients. So the final evaluation on the test set and on Hospital B should report the best model with and without ICULOS. If the gap grows on Hospital B, the clock was a shortcut. The 20% test set and Hospital B are kept untouched until then.